In [50]:
!pip install imbalanced-learn

In [55]:
import os
from tensorflow import keras
import tensorflow as tf
from keras import layers, models
import numpy as np
import kagglehub
import pandas as pd
from sklearn.utils import class_weight, shuffle
from imblearn.over_sampling import SMOTE
from keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report
import plotly.express as px
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [3]:
# Download latest version
path = kagglehub.dataset_download("shayanfazeli/heartbeat")

Using Colab cache for faster access to the 'heartbeat' dataset.


In [25]:
# Salva i percorsi dei file
train_csv_path = os.path.join(path, "mitbih_train.csv")
test_csv_path = os.path.join(path, "mitbih_test.csv")

In [26]:
# Carica i csv senza intestazione
# e converte i dati da testo in numeri (float32)
df_train = pd.read_csv(train_csv_path, header=None).astype("float32")
df_test = pd.read_csv(test_csv_path, header=None).astype("float32")

In [28]:
# MAPPING DELLE CLASSI
classes = ['N', 'S', 'V', 'F', 'Q']

In [29]:
# Imposto le features e le etichette/targets
X_train = df_train.iloc[:, :-1].values # features tutte le colonne tranne l'ultima
y_train = df_train.iloc[:, -1].values # target solo l'ultima colonna

In [30]:
# DATA AUGMENTATION CON SMOTE
# Le classi sono molto sbilanciate quindi uso SMOTE, un oversemplare
# che pareggia il numero di campioni nelle classi meno rappresentate
# 1. Generatore di dati sintetici
smote = SMOTE(random_state=42)

# 2. Genera i nuovi campioni bilanciati
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# 3. Mescola i dati
X_train_resampled, y_train_resampled = shuffle(X_train_resampled, y_train_resampled, random_state=42)

In [31]:
# converto in 3D per LSTM
X_train_3D = np.expand_dims(X_train_resampled, axis=2) # --> (87554, 187, 1)

In [32]:
# MODELLO
model = keras.Sequential([
    layers.Input(shape=(187, 1)), # 187 features e 1 sensore (battito)

    # LSTM Layer
    layers.LSTM(64, return_sequences=False), # False perchè dopo non c'è una LSTM

    # Layer di uscita per la regressione (1 valore: la temperatura del giorno dopo)
    layers.Dense(5, activation="softmax")
])

In [33]:
# COMPILAZIONE
# model is a classifier with 5 classes. it needs of optimizer, loss function and
# metric accuracy
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │           325 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,221 (67.27 KB)

 Trainable params: 17,221 (67.27 KB)

 Non-trainable params: 0 (0.00 B)

In [34]:
# Configurazione EarlyStopping
early_stop = EarlyStopping(
    monitor='val_loss',         # Controlla la loss sul validation set
    patience=4,                 # Aspetta 4 epoche di "secca" prima di fermarsi
    restore_best_weights=True,  # memorizza i pesi MIGLIORI (non gli ultimi)
    verbose=1                   # Stampa un messaggio in console quando si attiva
)

In [35]:
# TRAINING

hystory = model.fit(X_train_3D,
                    y_train_resampled,
                    epochs=30,
                    batch_size=32,
                    validation_split=0.2,
                    callbacks=[early_stop])

Epoch 1/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 106s 11ms/step - accuracy: 0.4595 - loss: 1.2554 - val_accuracy: 0.5218 - val_loss: 1.1423
Epoch 2/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 140s 11ms/step - accuracy: 0.5785 - loss: 1.0588 - val_accuracy: 0.6141 - val_loss: 0.9824
Epoch 3/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 109s 12ms/step - accuracy: 0.6342 - loss: 0.9666 - val_accuracy: 0.6700 - val_loss: 0.9072
Epoch 4/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 101s 11ms/step - accuracy: 0.6090 - loss: 0.9895 - val_accuracy: 0.6777 - val_loss: 0.8573
Epoch 5/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 101s 11ms/step - accuracy: 0.7038 - loss: 0.8131 - val_accuracy: 0.7363 - val_loss: 0.7252
Epoch 6/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 118s 13ms/step - accuracy: 0.7523 - loss: 0.6892 - val_accuracy: 0.7739 - val_loss: 0.6295
Epoch 7/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 115s 13ms/step - accuracy: 0.7877 - loss: 0.6005 - val_accuracy: 0.7926 - val_loss: 0.5845
Epoch 8/30
9059/9059 ━━━━━━━━━━━━━━━━━━━━ 107s 12ms/step - accuracy: 

In [36]:
# PREPARAZIONE TEST

# prende solo le features perchè anche nel test file di questo dataset
# c'è la colonna labels
X_test = df_test.iloc[:, :-1].values

# converto anche il test in 3D
X_test_3D = np.expand_dims(X_test, axis=2)

In [37]:
# CLASSIFICAZIONE
predictions = model.predict(X_test_3D)

685/685 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step


In [38]:
# ESTRAZIONE DATI DALLA PREDIZIONE

# calcola la probabilità e prende il massimo per riga (axis 1 cioè orizzontale)
predicted_classes = np.argmax(predictions, axis=1)

# converte le etichette numeriche in lettere comprensibili
predicted_labels = np.array(classes)[predicted_classes]

print("Classi previste per i primi 10 battiti:", predicted_labels[:10])

Classi previste per i primi 10 battiti: ['N' 'N' 'S' 'N' 'V' 'N' 'N' 'N' 'N' 'S']


In [58]:
y_test = df_test.iloc[:, -1].values
print(classification_report(y_test, predicted_classes, target_names=classes))

              precision    recall  f1-score   support

           N       0.99      0.89      0.93     18118
           S       0.30      0.81      0.44       556
           V       0.74      0.91      0.81      1448
           F       0.23      0.88      0.36       162
           Q       0.89      0.96      0.92      1608

    accuracy                           0.89     21892
   macro avg       0.63      0.89      0.69     21892
weighted avg       0.94      0.89      0.91     21892



In [57]:
# 1. Calcola le probabilità
real_values = np.argmax(y_test, axis=1) if len(y_test.shape) > 1 else y_test

# 2. Matrice di confusione in % (arrotondata a 2 decimali)
cm_percent = np.round(confusion_matrix(real_values, predicted_classes, normalize='true') * 100, 2)

# 3. Grafico Plotly
fig = px.imshow(
    cm_percent,
    x=classes,
    y=classes,
    color_continuous_scale='Blues',
    text_auto=True,  # Mostra le percentuali direttamente nelle caselle
    title="Matrice di Confusione Interattiva (%)"
)

fig.update_layout(xaxis_title="Predetto", yaxis_title="Reale")
fig.show()